# Anirban — NeuroMatch 2026 · Posterior Motives

**Your slide's questions:**

- Hierarchical model overview.
- Show the model produces **both** unimodal and bimodal responses.
- Use one clearly **unimodal** subject and one clearly **bimodal** subject.
- Compare what changes in the model between the two cases.
- Interpret how the model captures unimodality vs bimodality.

This notebook is runnable top-to-bottom and uses **HB-Rachel** (our hierarchical Bayesian observer) and the **Switch** (paper's switching observer). Edit `SUBJECT` / filters and re-run any cell. See the API guide cell for how to make your own calls.

---

> **Hierarchical Online model — trained subjects available now:** 2, 5, 8
> (model key `'hierarchical_online'`; more subjects can be trained via the CLI — see the API guide cell).
> Check which subjects are fit at any time with:
> ```python
> from observers import api
> api.fitted_subjects('hierarchical_online')   # -> [2, 5, 8]
> ```


In [ ]:
# ============================================================
#  SETUP  —  works in Google Colab or on a local checkout
# ============================================================
import os, sys, subprocess

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
BRANCH   = "model-verification"     # branch that carries the fitted models + API

def _find_root():
    """If we're already inside a checkout, use it (no clone needed)."""
    here = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "observers", "api.py")):
            return here
        here = os.path.dirname(here)
    return None

ROOT = _find_root()
if ROOT is None:
    # Colab path: shallow-clone the repo, then point at the hierarchical/ package.
    dest = "/content/NeuroMatch_2026_Behaviour" if os.path.isdir("/content") \
           else os.path.abspath("NeuroMatch_2026_Behaviour")
    if not os.path.isdir(dest):
        # PUBLIC repo: this just works. PRIVATE repo: replace REPO_URL with
        #   https://<TOKEN>@github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                        REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, "hierarchical")

sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from observers import api

print("repo root :", ROOT)
print("models    :", api.list_models())
print("data      :", "data/data01_direction4priors.csv  (12 subjects)")

# figures for this notebook go in a dedicated subfolder beside it
FIG_DIR = os.path.join(ROOT, "experiments", "anirban", "01_slide_notebook", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("figures  :", FIG_DIR)

## How to use the model API (read me — also for an LLM assistant)

Everything below comes from the **fitted models**, reached through one module:
`observers.api`. You never touch raw model math — you call functions.

### Where the data lives (relative to the repo's `hierarchical/` folder)
| what | path |
|---|---|
| trial-level data (all 12 subjects) | `data/data01_direction4priors.csv` |
| point fits (per model, per subject) | `results/fits/comparison/<model>/subject<N>.json` |
| cross-validation records | `results/fits/comparison_cv/<model>/subject<N>_cv.json` |
| project abstract | `docs/project_abstract.md` |

### Model keys
`'switch'` (paper's Switching observer, 9 params, non-learning) · `'hb_rachel'`
(our hierarchical Bayesian observer, 7 params, **learns** prior width online) ·
also available: `'hb_adaptive'`, `'hb_salma'`, `'recombined'`, `'basic_bayes'`,
and `'hierarchical_online'` (mixture-prior graded observer, 8 params, learns prior mean+width online).

**Which subjects are trained for a model?**
```python
api.fitted_subjects('hierarchical_online')   # -> list of subject IDs with a saved fit
```
To train a subject that isn't listed (writes a resumable fit to disk, ~20-90 min):
```bash
python -m observers.comparison.fit_batch --models hierarchical_online --subjects <N>
```

### The API — five kinds of call
```python
from observers import api

# LOAD --------------------------------------------------------------
api.load_subject(2)                 # -> DataFrame: motion_direction, motion_coherence,
                                    #    prior_std, estimate_dir  (one subject's trials)
api.load_fitted('hb_rachel', 2)     # -> (observer, record) with fitted params
api.observed_distribution(2, direction=335, coherence=0.06, prior_std=80)
                                    # -> empirical response histogram (density over 1..360)

# INSPECT -----------------------------------------------------------
api.list_models(); api.model_info() # what exists, params, colors
api.fitted_subjects('hb_rachel')    # which subjects are fit

# PREDICT -----------------------------------------------------------
api.predict('hb_rachel', 2)         # -> (n_trials, 360) predicted distribution per trial
api.belief_trajectory('hb_rachel', 2)
                                    # -> DataFrame trial, believed_sd  (the LEARNED prior width)

# FIT / SIMULATE ----------------------------------------------------
api.fit_model('hb_rachel', 2)       # refit from scratch (slow)
api.simulate('hb_rachel', 2, seed=0)# generative: synthetic responses from the fitted model

# EVALUATE ----------------------------------------------------------
api.results_table()                 # -> tidy DataFrame: model,label,subject,k,nll,aic,bic,cv_nll
api.trial_logliks('hb_rachel', 2)   # -> per-trial log-likelihood (slice it however you like)
api.bias_variability(2)             # -> per-condition estimation bias + circular SD (Fig-3 core)
```

### Custom calls (when a helper doesn't exist)
The API returns raw numbers; **you** aggregate/plot. To reach the model object
directly:
```python
from observers.comparison.registry import build_registry, load_subject
spec = build_registry(['hb_rachel'])['hb_rachel']   # a ModelSpec
obs, rec = api.load_fitted('hb_rachel', 2)
dists = spec.predict_distributions(obs, load_subject(2))  # (n_trials, 360)
```
Condition labels for any trial-level array come from `api.load_subject(s)` —
they're **row-aligned** with `predict`, `trial_logliks`, and `belief_trajectory`.

## 1 · The hierarchical observer in one picture

HB-Rachel holds a **mixed prior** — part a peaked von Mises bump at 225° (the informed prior), part a flat uniform (the naive prior) — mixed by a confidence weight **α**. Each trial it combines that prior with the sensory likelihood (width set by coherence) and reads out an estimate. Whether the response distribution is **one peak or two** is set by how sharp the evidence is: broad evidence (low coherence) leaves the prior bump standing next to the stimulus → **bimodal**; sharp evidence (high coherence) overwhelms the prior → **unimodal**.

In [ ]:
# The demonstration: bimodal -> unimodal is NOT a property of certain subjects,
# it's what ONE graded observer does as sensory evidence sharpens. We hold subject,
# stimulus, and prior block FIXED and vary ONLY coherence.
#
# Config (subject must be fit for all three overlay models; hierarchical_online is
# currently fit for 2, 5, 8). Subject 2 at stimulus 325deg (far from the 225deg prior),
# wide prior block (SD 80deg) has trials at all three coherence levels.
SUBJ, STIM, PSTD = 2, 325, 80
COHS = [0.06, 0.12, 0.24]

# sanity: show trial counts per coherence in this cell
df = api.load_subject(SUBJ)
_d, _c, _ps = df.motion_direction.values, df.motion_coherence.values, df.prior_std.values
for coh in COHS:
    n = int(((_d==STIM)&(_c==coh)&(_ps==PSTD)).sum())
    print(f'coherence {coh:g}: n={n} trials  (subject {SUBJ}, stim {STIM}deg, prior SD {PSTD}deg)')

## 2 · Bimodal → unimodal as coherence rises (one graded observer, no switch)

The core claim in one figure. Same subject, same far stimulus (325°, well away from the 225° prior), same wide-prior block — **only coherence changes**:

- **Low coherence (0.06):** evidence is weak, so the prior competes with it — responses split between the prior (225°) and the stimulus → **bimodal**.
- **Medium coherence (0.12):** evidence strengthens; mass shifts toward the stimulus.
- **High coherence (0.24):** evidence dominates; responses collapse onto the stimulus → **unimodal**.

No discrete switch is invoked anywhere — the transition is the smooth consequence of graded Bayesian integration as the likelihood sharpens. Three fitted models are overlaid: **Hierarchical Online** (green, focal), **Switch** (blue), **HB-Rachel** (orange). Observed responses (grey) are von-Mises-smoothed so the clusters read as clusters; n per panel is small (see cell above), so lean on the model curves — they use the full belief replay, not just this cell's trials.

In [ ]:
def vm_smooth(density, kappa=50.0):
    ang = np.deg2rad(np.arange(360)); k = np.exp(kappa*np.cos(ang-ang[0])); k /= k.sum()
    out = np.real(np.fft.ifft(np.fft.fft(np.asarray(density,float)) * np.fft.fft(k)))
    return out / out.sum()

MODELS = [('hierarchical_online', 'Hierarchical Online', '#2e7d32', 2.6, 1.0),
          ('switch',              'Switch',              '#1f6f8b', 1.7, 0.9),
          ('hb_rachel',           'HB-Rachel',           '#e08214', 1.7, 0.9)]
DEG = np.arange(1, 361)
preds = {k: api.predict(k, SUBJ) for k,*_ in MODELS}
d, c, ps = df.motion_direction.values, df.motion_coherence.values, df.prior_std.values

fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), sharex=True)
for ax, coh in zip(axes, COHS):
    m = (d==STIM) & (c==coh) & (ps==PSTD); n = int(m.sum())
    obs = vm_smooth(api.observed_distribution(SUBJ, direction=STIM, coherence=coh, prior_std=PSTD), kappa=50)
    ax.fill_between(DEG, obs, color='#9a9a9a', alpha=0.35, lw=0, zorder=1, label=f'observed (n={n})')
    for key, label, col, lw, a in MODELS:
        pr = preds[key][m].mean(0); pr = pr/pr.sum()
        ax.plot(DEG, vm_smooth(pr, kappa=200), color=col, lw=lw, alpha=a, zorder=3, label=label, solid_capstyle='round')
    ax.axvline(225, ls=':', c='k', lw=1.1, alpha=0.55, zorder=2)
    ax.axvline(STIM, ls='--', c='#c0392b', lw=1.1, alpha=0.6, zorder=2)
    ax.set_xlim(160, 360); ax.set_ylim(0, None); ax.margins(y=0.12)
    ax.set_title(f'coherence {coh:g}', loc='left'); ax.set_xlabel('estimated direction (°)')
    for sp in ('top','right'): ax.spines[sp].set_visible(False)
    yb = ax.get_ylim()[1]*0.03
    ax.text(225, yb, 'prior', rotation=90, va='bottom', ha='right', fontsize=7, color='k', alpha=0.6)
    ax.text(STIM, yb, 'stimulus', rotation=90, va='bottom', ha='right', fontsize=7, color='#c0392b', alpha=0.8)
axes[0].set_ylabel('response density')
axes[2].legend(loc='upper left', frameon=False, fontsize=7)
fig.suptitle(f'Bimodal → unimodal as coherence rises  ·  one graded observer, no switch\n'
             f'subject {SUBJ}, stimulus {STIM}° (far from prior), wide prior block (SD 80°) — only coherence varies; '
             f'observed = von-Mises-smoothed responses', fontsize=8.5, y=1.06, ha='center')
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'anirban_bimodality.png'), dpi=150, bbox_inches='tight'); plt.show()

## 3 · What drives the transition — and the honest gap

The bimodal→unimodal shift above is driven by **coherence alone** (subject, stimulus, and prior block are held fixed). In the model, coherence enters only the sensory likelihood width `k_llh`: low coherence → broad likelihood that the prior can compete with → two clusters; high coherence → sharp likelihood that dominates → one cluster at the stimulus. The cell below quantifies the cluster masses across the sweep, and flags the honest gap: at the low/medium coherences all three models **under-represent the stimulus cluster** relative to the observed data (the signature of averaging the readout, which smears the stimulus lobe into the gap between the peaks).

In [ ]:
import numpy as np
DEG = np.arange(1, 361)
def cd(a,b): return np.abs((a-b+180)%360-180)
def cmass(dist, stim): return (round(float(dist[cd(DEG,225)<=25].sum()),3), round(float(dist[cd(DEG,stim)<=25].sum()),3))

print(f'cluster mass (prior@225°, stimulus@{STIM}°) across the coherence sweep')
print(f'subject {SUBJ}, wide prior block (SD 80°):\n')
d, c, ps = df.motion_direction.values, df.motion_coherence.values, df.prior_std.values
for coh in COHS:
    m = (d==STIM) & (c==coh) & (ps==PSTD)
    obs = np.asarray(api.observed_distribution(SUBJ, direction=STIM, coherence=coh, prior_std=PSTD), float); obs/=obs.sum()
    print(f'coherence {coh:g}  (n={int(m.sum())}):')
    print(f'    observed             {cmass(obs, STIM)}')
    for k, l in [('hierarchical_online','Hierarchical Online'),('switch','Switch'),('hb_rachel','HB-Rachel')]:
        p = api.predict(k, SUBJ)[m].mean(0); p/=p.sum()
        print(f'    {l:20s} {cmass(p, STIM)}')
    print()
print('Read: stimulus-cluster mass rises with coherence for observed AND all three models')
print('-> the bimodal->unimodal transition. The gap: at low/mid coherence the models put')
print('LESS mass at the stimulus than the data does (Switch also over-weights the prior;')
print('Hierarchical Online is broad, under-filling both peaks into the middle).')

**Interpretation (abstract claim i), stated honestly.**

1. **Bimodality is a graded phenomenon, not a switch.** Holding the subject and stimulus fixed and sweeping only coherence, the response goes from two clusters (prior + stimulus) at low coherence to one cluster (stimulus) at high coherence. This emerges in the fully graded **Hierarchical Online** observer — whose `sample` readout is probability-matching over the responsibility-weighted posterior, with **no discrete switch operator**. So the two-cluster shape and its disappearance are both consequences of Bayesian integration as the likelihood sharpens; they do not require a prior-vs-evidence switch. That is the claim-i demonstration.

2. **But the models under-represent the stimulus cluster.** At low/medium coherence the observed data already carries substantial mass at the stimulus (e.g. at coherence 0.12, observed stimulus mass ≈0.46 vs ≈0.18–0.23 for every model), so all three fall short of the observed stimulus lobe. The pattern at the prior differs by model — Switch over-weights the prior, while Hierarchical Online is broad and under-fills *both* peaks (spreading mass into the gap between them). The common, robust miss is the **under-filled stimulus cluster** — the signature of averaging the readout, which smears the stimulus lobe. This is a real limitation to flag, not paper over.

**Bottom line for the slide.** The graded-Bayesian account reproduces the *qualitative* coherence-driven bimodal→unimodal transition without any switch mechanism (claim i holds at the level of shape), but none of the current readouts reproduces the observed *balance* between clusters at low coherence — they under-represent the stimulus. Whether to present this as "graded inference is sufficient" (with the weighting gap as future work) or to first refine the readout is a modelling decision — surfaced here, not decided.